# Detector de somnolencia/fatiga al volante
## Notebook 02 (v2) — Entrenamiento, evaluación y exportación de la CNN de estado ocular

Clasificador binario de **ojo cerrado (1) / ojo abierto (0)** a partir de recortes de 64 × 64 píxeles en escala de grises (MRL Eye Dataset), con partición por persona.

**Características de esta versión**
- La clase positiva es **ojo cerrado**; la salida de la red es la probabilidad de ojo cerrado (`p_cerrado`).
- El escalado de intensidades (división entre 255) forma parte del modelo, de modo que la aplicación entrega recortes con valores 0–255 sin preprocesamiento adicional.
- Aumento de datos solo en entrenamiento (volteo, rotación, desplazamiento, zoom, brillo, contraste, ruido y desenfoque), para mejorar la tolerancia a cámaras y condiciones de iluminación distintas a las del conjunto original.
- Ponderación de clases (`class_weight`) y detención temprana con el conjunto de validación.
- Validación cruzada agrupada por persona (5 pliegues).
- Evaluación única en el conjunto de prueba: exactitud, precisión, sensibilidad, especificidad, F1, matriz de confusión y curva ROC, en total y por uso de lentes.
- Exportación de pesos para la aplicación web.

**Requisitos en Google Drive:** `MyDrive/ProyectoSomnolencia/datos/` con `mrl_train.npz`, `mrl_val.npz`, `mrl_test.npz` y `particion_mrl.csv` (salidas del Notebook 01).
**Entorno:** GPU (Entorno de ejecución → Cambiar tipo de entorno → T4).

## 1. Configuración y datos

In [ ]:
import os, json, time, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from google.colab import drive

SEED = 42
keras.utils.set_random_seed(SEED)
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

drive.mount('/content/drive')
import glob
# Busca la carpeta de datos: directa en Mi unidad o como acceso directo a una carpeta compartida
candidatas = ['/content/drive/MyDrive/ProyectoSomnolencia'] + glob.glob('/content/drive/MyDrive/.shortcut-targets-by-id/*/ProyectoSomnolencia')
BASE = next((c for c in candidatas if os.path.exists(c + '/datos/mrl_train.npz')), None)
assert BASE, 'No se encontró ProyectoSomnolencia/datos en Drive. Añada un acceso directo a Mi unidad (ver guía).'
print('Carpeta de datos:', BASE)
RUTA_DATOS = BASE + '/datos'
SALIDA = 'resultados_entrenamiento'
os.makedirs(SALIDA + '/web_modelo', exist_ok=True)

EPOCAS = 30
BATCH = 128
EJECUTAR_CV = True        # validación cruzada de 5 pliegues (aprox. 10 min adicionales)

tr, va, te = (np.load(f'{RUTA_DATOS}/mrl_{s}.npz') for s in ('train', 'val', 'test'))
Xtr, Xva, Xte = tr['X'], va['X'], te['X']
# En los .npz y = 1 (abierto) / 0 (cerrado). Aquí se invierte: 1 = CERRADO (clase positiva).
ytr, yva, yte = (1 - tr['y']).astype('float32'), (1 - va['y']).astype('float32'), (1 - te['y']).astype('float32')
fold_dev = np.concatenate([tr['fold'], va['fold']])

part = pd.read_csv(RUTA_DATOS + '/particion_mrl.csv')
lentes_te = part[part.split == 'test'].lentes.to_numpy()
assert len(lentes_te) == len(yte)

for nombre, X, y in [('Entrenamiento', Xtr, ytr), ('Validación', Xva, yva), ('Prueba', Xte, yte)]:
    print(f'{nombre:14s}: {X.shape} | cerrados: {100 * y.mean():.1f} %')

## 2. Aumento de datos
Se aplica **solo al entrenamiento**. Además de transformaciones geométricas leves, se varían brillo, contraste, ruido y desenfoque, porque las imágenes que recibirá la aplicación (cámara web o celular, a color y convertidas a gris) difieren de las imágenes infrarrojas del MRL.

In [ ]:
aug = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.3, value_range=(0, 255)),
    layers.RandomContrast(0.3),
], name='aumento')

def desenfoque_aleatorio(x, prob=0.3):
    """Desenfoque medio 3x3 aplicado a una fracción de las imágenes del lote."""
    borroso = tf.nn.avg_pool2d(x, ksize=3, strides=1, padding='SAME')
    mascara = tf.random.uniform([tf.shape(x)[0], 1, 1, 1]) < prob
    return tf.where(mascara, borroso, x)

def preparar_train(x, y):
    x = tf.cast(x, tf.float32)
    x = aug(x, training=True)
    x = x + tf.random.normal(tf.shape(x), stddev=6.0)   # ruido gaussiano (escala 0-255)
    x = desenfoque_aleatorio(x)
    return tf.clip_by_value(x, 0.0, 255.0), y

def preparar_eval(x, y):
    return tf.cast(x, tf.float32), y

def dataset(X, y, entrenamiento):
    ds = tf.data.Dataset.from_tensor_slices((X[..., None], y))
    if entrenamiento:
        ds = ds.shuffle(10_000, seed=SEED, reshuffle_each_iteration=True).batch(BATCH).map(preparar_train, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.batch(512).map(preparar_eval, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

# Ejemplos de aumento (para el informe)
lote_x, lote_y = next(iter(dataset(Xtr[:512], ytr[:512], True).take(1)))
fig, ax = plt.subplots(2, 8, figsize=(14, 4))
for j in range(8):
    ax[0, j].imshow(Xtr[j], cmap='gray', vmin=0, vmax=255); ax[0, j].axis('off')
    ax[1, j].imshow(lote_x[j, ..., 0], cmap='gray', vmin=0, vmax=255); ax[1, j].axis('off')
ax[0, 0].set_title('Originales', loc='left', fontsize=10); ax[1, 0].set_title('Con aumento', loc='left', fontsize=10)
plt.tight_layout(); plt.savefig(SALIDA + '/fig_aumento_datos.png', dpi=150); plt.show()

## 3. Arquitectura
CNN ligera: tres bloques de convolución y agrupamiento, una capa densa con *dropout* y una neurona de salida con sigmoide. La **capa de reescalado (÷255) está dentro del modelo**, por lo que la entrada son recortes de 64 × 64 con valores 0–255.

In [ ]:
def construir_modelo():
    return keras.Sequential([
        layers.Input(shape=(64, 64, 1)),
        layers.Rescaling(1.0 / 255),
        layers.Conv2D(16, 3, padding='same', activation='relu'), layers.MaxPooling2D(),
        layers.Conv2D(32, 3, padding='same', activation='relu'), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding='same', activation='relu'), layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid'),    # p_cerrado
    ], name='cnn_estado_ocular')

def compilar(m):
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy',
              metrics=['accuracy', keras.metrics.Recall(name='sensibilidad'), keras.metrics.Precision(name='precision')])
    return m

def callbacks(paciencia=6):
    return [keras.callbacks.EarlyStopping(monitor='val_loss', patience=paciencia, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)]

def pesos_clase(y):
    w = compute_class_weight('balanced', classes=np.array([0.0, 1.0]), y=y)
    return {0: float(w[0]), 1: float(w[1])}

modelo = compilar(construir_modelo())
modelo.summary()

## 4. Entrenamiento
Pérdida de entropía cruzada binaria, optimizador Adam, `class_weight` calculado con el conjunto de entrenamiento y detención temprana según la pérdida de validación (se restauran los mejores pesos).

In [ ]:
cw = pesos_clase(ytr)
print('Pesos de clase (0 = abierto, 1 = cerrado):', cw)

t0 = time.time()
hist = modelo.fit(dataset(Xtr, ytr, True), validation_data=dataset(Xva, yva, False),
                  epochs=EPOCAS, class_weight=cw, callbacks=callbacks(), verbose=2)
tiempo_min = (time.time() - t0) / 60
n_epocas = len(hist.history['loss'])
print(f'\nÉpocas ejecutadas: {n_epocas} | tiempo: {tiempo_min:.1f} min')

h = pd.DataFrame(hist.history); h.index += 1
h.rename_axis('epoca').to_csv(SALIDA + '/historial_entrenamiento.csv')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h.index, h['loss'], label='Entrenamiento'); ax[0].plot(h.index, h['val_loss'], label='Validación')
ax[0].set_xlabel('Época'); ax[0].set_ylabel('Pérdida'); ax[0].set_title('Curva de pérdida'); ax[0].legend()
ax[1].plot(h.index, h['accuracy'], label='Entrenamiento'); ax[1].plot(h.index, h['val_accuracy'], label='Validación')
ax[1].set_xlabel('Época'); ax[1].set_ylabel('Exactitud'); ax[1].set_title('Curva de exactitud'); ax[1].legend()
plt.tight_layout(); plt.savefig(SALIDA + '/fig_curvas_entrenamiento.png', dpi=150); plt.show()

## 5. Evaluación
### 5.1 Umbral de decisión (elegido con validación)
Se calcula el umbral que maximiza el índice de Youden (sensibilidad + especificidad − 1) sobre el conjunto de validación. El conjunto de prueba **no** interviene en esta elección.

In [ ]:
def predecir(m, X):
    return m.predict(dataset(X, np.zeros(len(X), 'float32'), False).map(lambda x, y: x), verbose=0).ravel()

def metricas(y, p, umbral=0.5):
    pred = (p >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    prec = tp / (tp + fp) if tp + fp else float('nan')
    sens = tp / (tp + fn) if tp + fn else float('nan')
    espe = tn / (tn + fp) if tn + fp else float('nan')
    f1 = 2 * prec * sens / (prec + sens) if prec + sens else float('nan')
    return {'exactitud': (tp + tn) / len(y), 'precision': prec, 'sensibilidad': sens, 'especificidad': espe,
            'f1': f1, 'vp': int(tp), 'fp': int(fp), 'fn': int(fn), 'vn': int(tn), 'n': int(len(y))}

pva = predecir(modelo, Xva)
fpr_v, tpr_v, thr_v = roc_curve(yva, pva)
umbral_val = float(thr_v[np.argmax(tpr_v - fpr_v)])
print(f'Umbral de validación (Youden): {umbral_val:.3f}')

### 5.2 Conjunto de prueba (evaluación única)

In [ ]:
pte = predecir(modelo, Xte)
res = {'umbral_validacion': umbral_val, 'auc_prueba': float(roc_auc_score(yte, pte)),
       'prueba_umbral_0.5': metricas(yte, pte, 0.5),
       'prueba_umbral_validacion': metricas(yte, pte, umbral_val)}
for g, nombre in [(0, 'sin_lentes'), (1, 'con_lentes')]:
    m = lentes_te == g
    if m.sum():
        res['prueba_' + nombre] = metricas(yte[m], pte[m], 0.5)

filas = [{'conjunto': k.replace('prueba_', 'Prueba: '), **v} for k, v in res.items() if isinstance(v, dict)]
tabla = pd.DataFrame(filas).set_index('conjunto')
tabla.to_csv(SALIDA + '/tabla_metricas_prueba.csv', encoding='utf-8-sig')
print('AUC en prueba: %.4f' % res['auc_prueba'])
pd.set_option('display.float_format', lambda v: f'{v:.3f}')
tabla

In [ ]:
m = res['prueba_umbral_0.5']
cm = np.array([[m['vn'], m['fp']], [m['fn'], m['vp']]])
fpr, tpr, _ = roc_curve(yte, pte)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
ax[0].imshow(cm, cmap='Blues')
ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(['Abierto', 'Cerrado']); ax[0].set_yticks([0, 1]); ax[0].set_yticklabels(['Abierto', 'Cerrado'])
ax[0].set_xlabel('Predicción'); ax[0].set_ylabel('Real'); ax[0].set_title('Matriz de confusión (prueba, umbral 0,5)')
for i in range(2):
    for j in range(2):
        ax[0].text(j, i, f'{cm[i, j]:,}'.replace(',', ' '), ha='center', va='center',
                   color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=12)
ax[1].plot(fpr, tpr, lw=2, label=f'AUC = {res["auc_prueba"]:.3f}'); ax[1].plot([0, 1], [0, 1], '--', color='gray')
ax[1].set_xlabel('1 − especificidad'); ax[1].set_ylabel('Sensibilidad'); ax[1].set_title('Curva ROC (clase positiva: ojo cerrado)'); ax[1].legend(loc='lower right')
plt.tight_layout(); plt.savefig(SALIDA + '/fig_matriz_roc.png', dpi=150); plt.show()

## 6. Validación cruzada agrupada por persona (5 pliegues)
Se entrena un modelo nuevo por pliegue con entrenamiento + validación, usando la asignación de pliegues del Notebook 01 (ninguna persona aparece en el entrenamiento y la validación de un mismo pliegue). El conjunto de prueba no interviene.

In [ ]:
cv_filas = []
if EJECUTAR_CV:
    X_dev = np.concatenate([Xtr, Xva]); y_dev = np.concatenate([ytr, yva])
    for k in range(1, 6):
        idx_va, idx_tr = fold_dev == k, fold_dev != k
        mk = compilar(construir_modelo())
        mk.fit(dataset(X_dev[idx_tr], y_dev[idx_tr], True), validation_data=dataset(X_dev[idx_va], y_dev[idx_va], False),
               epochs=12, class_weight=pesos_clase(y_dev[idx_tr]), callbacks=callbacks(3), verbose=0)
        mm = metricas(y_dev[idx_va], predecir(mk, X_dev[idx_va]), 0.5)
        cv_filas.append({'Pliegue': k, 'Imágenes de validación': mm['n'], 'Exactitud': mm['exactitud'],
                         'Sensibilidad': mm['sensibilidad'], 'Especificidad': mm['especificidad'], 'F1': mm['f1']})
        print(f'Pliegue {k}: exactitud {mm["exactitud"]:.3f} | sensibilidad {mm["sensibilidad"]:.3f} | especificidad {mm["especificidad"]:.3f}')
        keras.backend.clear_session()
    cv = pd.DataFrame(cv_filas).set_index('Pliegue')
    resumen_cv = cv[['Exactitud', 'Sensibilidad', 'Especificidad', 'F1']].agg(['mean', 'std']).round(4)
    cv.to_csv(SALIDA + '/tabla_validacion_cruzada.csv', encoding='utf-8-sig')
    res['validacion_cruzada'] = {c: {'media': float(resumen_cv.loc['mean', c]), 'desv': float(resumen_cv.loc['std', c])} for c in resumen_cv.columns}
    display(cv); display(resumen_cv)
else:
    print('Validación cruzada omitida (EJECUTAR_CV = False).')

## 7. Exportación del modelo
- Modelo Keras en Google Drive (`eye_cnn_v2.keras`).
- Pesos en formato binario `float32` (`pesos.bin`) y descriptor `modelo_meta.json` para la aplicación web.
- `referencia.json`: imágenes de prueba con la salida exacta del modelo, para verificar que la versión JavaScript reproduce los mismos valores.

In [ ]:
try:
    modelo.save('/content/drive/MyDrive/eye_cnn_v2.keras')
    print('Modelo guardado en Mi unidad: eye_cnn_v2.keras')
except Exception as e:
    modelo.save(SALIDA + '/eye_cnn_v2.keras')
    print('No se pudo guardar en Drive (', e, '). El modelo quedó dentro del zip.')

pesos = modelo.get_weights()
with open(SALIDA + '/web_modelo/pesos.bin', 'wb') as f:
    for w in pesos:
        f.write(np.ascontiguousarray(w, dtype='<f4').tobytes())

meta = {
    'entrada': [64, 64, 1],
    'valores_entrada': '0-255 (la división entre 255 forma parte del modelo)',
    'salida': 'p_cerrado',
    'umbral_validacion': umbral_val,
    'capas': [{'tipo': l.__class__.__name__, 'nombre': l.name} for l in modelo.layers],
    'tensores': [{'forma': list(w.shape)} for w in pesos],
}
with open(SALIDA + '/web_modelo/modelo_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

ref = {'imagenes': Xte[:5].reshape(5, -1).tolist(), 'etiqueta_cerrado': yte[:5].tolist(),
       'p_cerrado': predecir(modelo, Xte[:5]).tolist()}
with open(SALIDA + '/web_modelo/referencia.json', 'w') as f:
    json.dump(ref, f)
print('pesos.bin: %.2f MB' % (os.path.getsize(SALIDA + '/web_modelo/pesos.bin') / 1e6))

## 8. Resumen y descarga
**Copiar el bloque impreso abajo y pegarlo en el chat**, o enviar el archivo `resultados_entrenamiento.zip`.

In [ ]:
def redondear(o):
    if isinstance(o, dict): return {k: redondear(v) for k, v in o.items()}
    if isinstance(o, float): return round(o, 4)
    return o

resumen = redondear({'epocas_ejecutadas': n_epocas, 'tiempo_entrenamiento_min': round(tiempo_min, 2),
                     'pesos_de_clase': cw, 'parametros': int(modelo.count_params()), **res})
with open(SALIDA + '/resumen.json', 'w') as f:
    json.dump(resumen, f, indent=2)
print(json.dumps(resumen, indent=2))

with zipfile.ZipFile('resultados_entrenamiento.zip', 'w') as z:
    for raiz, _, archivos in os.walk(SALIDA):
        for a in archivos:
            z.write(os.path.join(raiz, a))
from google.colab import files
files.download('resultados_entrenamiento.zip')